# 05 — Perennial vs annual vs pasture/fallow: trends in Piura

Figures for `docs/perennial/RESULTS.md`. Follows the existing notebook conventions: paths are
relative to `notebooks/` (i.e. `../data/processed/…`), and every satellite time-series figure
marks the Landsat mission boundaries so a sensor step is never read as land-use change.

**Run order.** This notebook only *reads* artifacts; produce them first with

```bash
export CC_PROC=data/processed/perennial CC_RUNS=runs/perennial
uv run python -m crop_classifier.cli perennial labels
uv run python -m crop_classifier.cli splits assign
uv run python -m crop_classifier.cli features assemble
uv run python -m crop_classifier.cli train --model rules|lightgbm|ltae|psetae
uv run python -m crop_classifier.cli perennial panel build|extract|assemble|infer --run <run>
uv run python -m crop_classifier.cli perennial mapbiomas
```

In [ ]:
import json, os, sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
os.environ.setdefault("CC_PROC", "../data/processed/perennial")
os.environ.setdefault("CC_RUNS", "../runs/perennial")

from crop_classifier.perennial import analysis, area, figures, mapbiomas, trajectories

PROC = Path(os.environ["CC_PROC"])
RUNS = Path(os.environ["CC_RUNS"])
FIG = Path("../docs/perennial/figures"); FIG.mkdir(parents=True, exist_ok=True)
CLASSES = ["ANNUAL", "PASTURE_FALLOW", "PERENNIAL"]
plt.rcParams.update({"figure.dpi": 120, "font.size": 9})

## 1. Label space and spatial splits

What the 3-class map is made of, and how much spatial autocorrelation the split has to fight.

In [ ]:
parcels = gpd.read_parquet(PROC / "modeling_parcels.parquet")
print(f"{len(parcels):,} parcels")
display(parcels["label"].value_counts().rename("n_parcels").to_frame()
        .assign(share=lambda d: d.n_parcels / len(parcels)))

meta = json.load(open(PROC / "splits_meta.json"))
curve = pd.DataFrame(meta["audit"]["curve"]).dropna(subset=["agreement"])
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(curve["hi_m"], curve["agreement"], "o-")
ax.axhline(meta["audit"]["baseline_agreement"], ls="--", c="k", label="random baseline")
ax.axvline(meta["config"]["buffer_m"], ls=":", c="r", label="buffer_m")
ax.set(xlabel="distance (m)", ylabel="neighbour class agreement",
       title="spatial autocorrelation (3 classes)")
ax.legend(fontsize=8); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(FIG / "autocorrelation.png"); plt.show()

## 2. Model comparison — pooled spatial CV

Four models on identical folds: the phenology **rule** (the scientific control), LightGBM, LTAE, PSE-LTAE. Every number is pooled *validation*; the locked test set is spent once, separately.

In [ ]:
comp = pd.read_csv(RUNS / "comparison" / "model_comparison.csv")
display(comp.round(3))
per_class = pd.read_csv(RUNS / "comparison" / "per_class_f1.csv")
display(per_class.pivot(index="class", columns="model", values="f1").round(3))

for img in ("pooled_cv_metrics.png", "per_class_f1.png", "cv_macro_f1_by_fold.png"):
    p = RUNS / "comparison" / img
    if p.exists():
        display(__import__("IPython").display.Image(str(p)))

### 2b. The rule model's decision boundary

Thresholds fitted per fold on training data only. **Stability of the thresholds across the five spatial folds is itself a result**: stable ⇒ the rule generalises.

In [ ]:
rows = []
for k in range(5):
    f = RUNS / "rules_3c" / f"fold{k}" / "thresholds.json"
    if f.exists():
        rows.append({"fold": k, **json.load(open(f))})
thr = pd.DataFrame(rows)
display(thr[["fold", "NDVI_level_hi", "NDVI_amp_max", "NDVI_peak_min", "train_macro_f1"]]
        .round(3))
print("threshold spread (max-min):")
print((thr[["NDVI_level_hi", "NDVI_amp_max", "NDVI_peak_min"]].max()
       - thr[["NDVI_level_hi", "NDVI_amp_max", "NDVI_peak_min"]].min()).round(3))
p = RUNS / "rules_3c" / "fold0" / "rule_boundary.png"
if p.exists(): display(__import__("IPython").display.Image(str(p)))

## 3. Calibration

Temperature scaling (D8) — argmax-preserving, so it moves confidences only. Required before any confidence threshold or probability-weighted area estimate means anything.

In [ ]:
sel = json.load(open(RUNS / "selected_model.json"))["run"] if (RUNS / "selected_model.json").exists() else "lightgbm_3c"
t = json.load(open(RUNS / sel / "temperature.json"))
print(f"selected model: {sel}")
print(f"T = {t['temperature']:.3f}   NLL {t['nll_before']:.4f} -> {t['nll_after']:.4f}"
      f"   ECE {t['ece_before']:.4f} -> {t['ece_after']:.4f}")
display(__import__("IPython").display.Image(str(RUNS / sel / "reliability.png")))

## 4. Locked test set — spent once

Everything from here is *reported*, not tuned. The confusion matrix below is the error matrix for the Olofsson area correction.

In [ ]:
tm = json.load(open(RUNS / sel / "test_metrics.json"))
display(pd.Series(tm).to_frame("value").round(3))
display(__import__("IPython").display.Image(str(RUNS / sel / "confusion.png")))
display(pd.read_csv(RUNS / sel / "per_class.csv").round(3))

## 5. Sensor-shift diagnostics (§7.4)

Steps at 1999 / 2012 / 2013 are sensor artefacts. OLI (L8/L9) is harmonised to ETM+ with the Roy et al. (2016) coefficients at assembly time; these plots are the check that it worked. **2012 is L7-SLC-off only — flagged, never interpolated over.**

In [ ]:
drift = pd.read_csv(PROC / "panel_feature_drift.csv")
figures.feature_drift_figure(drift, FIG / "feature_drift.png")
display(__import__("IPython").display.Image(str(FIG / "feature_drift.png")))

## 6. Temporal transfer (§7.3, criterion S4)

Labels exist only for ~1996–2007 but we predict 2024. Accuracy against the label-year class as a function of distance *k* is a **lower bound** on model stability — real land-use change also contributes to the decay.

In [ ]:
tt = pd.read_csv(PROC / "temporal_transfer.csv")
display(tt.round(3))
figures.temporal_transfer_figure(tt, FIG / "temporal_transfer.png")
display(__import__("IPython").display.Image(str(FIG / "temporal_transfer.png")))

## 7. Trajectories and change detection (§9)

A transition is recorded only when the new class persists **≥3 consecutive observed years** — orchard establishment and removal are multi-year events, so a 1-year excursion is noise by construction.

In [ ]:
panel_preds = pd.read_parquet(PROC / "panel_predictions.parquet")
res = analysis.run(panel_preds, CLASSES, save=False)
display(res["flicker"].round(3))
display(res["transition_matrix"])
figures.transition_matrix_figure(res["transition_matrix"], FIG / "transition_matrix.png")
display(__import__("IPython").display.Image(str(FIG / "transition_matrix.png")))

In [ ]:
figures.establishment_ramp_figure(panel_preds, res["transitions"],
                                  FIG / "establishment_ramp.png")
p = FIG / "establishment_ramp.png"
if p.exists(): display(__import__("IPython").display.Image(str(p)))

panel_parcels = gpd.read_parquet(PROC / "panel_parcels.parquet")
figures.transition_map_figure(panel_parcels, res["transitions"], FIG / "transition_map.png")
display(__import__("IPython").display.Image(str(FIG / "transition_map.png")))

## 8. Area share 1990–2024 with uncertainty (§9.4)

Three estimators: naive argmax, probability-weighted, and **Olofsson et al. (2014) bias-corrected with 95 % CIs** from the locked-test error matrix. The CIs are the point — an uncertainty-free trend line from a macro-F1 ≈ 0.7 classifier is not publishable.

In [ ]:
trend = pd.read_csv(PROC / "area_trend.csv")
share = pd.read_csv(PROC / "class_share_by_year.csv")
ol = trend[trend.method == "olofsson2014"]
figures.area_share_figure(share, FIG / "area_share.png", ci=ol)
display(__import__("IPython").display.Image(str(FIG / "area_share.png")))
display(analysis.perennial_share_trend(share))

## 9. MapBiomas Peru benchmark (§8)

**Independence caveat:** MapBiomas Peru is itself Landsat-derived at 30 m, so it shares sensors, cloud regimes and mixed-pixel problems with our model. It is a benchmark, not ground truth.

In [ ]:
mb = pd.read_parquet(PROC / "mapbiomas_panel.parquet")
hist = mapbiomas.code_histogram(mb)
display(hist.head(15))
print("\nCodes for perennial crop (36/46/47/48) and pasture (15):")
display(hist[hist.mb_code.isin([36, 46, 47, 48, 15])])

In [ ]:
rows = []
for mosaic in (None, "ANNUAL", "PERENNIAL", "PASTURE_FALLOW"):
    for purity in (0.0, 0.6):
        rows.append(mapbiomas.agreement_vs_pett(
            mb, parcels[parcels.split == "test"], mosaic_as=mosaic, min_purity=purity))
display(pd.DataFrame(rows).round(3))
print("criterion S3: compare the best MapBiomas macro-F1 above with our locked-test "
      "macro-F1 in section 4.")

## 10. Sensitivity battery (§9.5)

A trend that survives all five perturbations is a finding; a trend that flips under any of them is an artefact.

In [ ]:
variants = {"headline (smoothed, ML, cana=annual)": share}
p = PROC / "sensitivity_shares.json"
if p.exists():
    for name, rec in json.load(open(p)).items():
        variants[name] = pd.DataFrame(rec)
display(analysis.sensitivity_table(variants).round(4))